# 1️⃣ Introducción

Este cuaderno construye un modelo sencillo para predecir si un equipo gana (1) o pierde (0) un partido de la NBA usando únicamente estadísticas **ROLL10_** (promedios móviles de los últimos 10 juegos).

- Cada fila representa el rendimiento de un equipo en un partido, sin diferenciar condición de local o visitante.
- El usuario puede elegir una fecha de corte para entrenar el modelo con la historia previa y evaluar el rendimiento a partir de esa fecha.

## 2️⃣ Librerías e imports

Se importan las librerías necesarias para análisis, modelado y widgets interactivos.

In [ ]:
import datetime as dt

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.calibration import CalibrationDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import ipywidgets as widgets
from IPython.display import display

sns.set(style="whitegrid")

## 3️⃣ Carga y limpieza de datos

Se carga el parquet con los gamelogs de equipos, se seleccionan únicamente las columnas ROLL10_, la fecha y el objetivo `WL_NUM`, y se eliminan filas con valores nulos.

In [ ]:
DATA_PATH = "/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/teamgamelogs_by_game.parquet"

raw_df = pd.read_parquet(DATA_PATH)
raw_df['GAME_DATE'] = pd.to_datetime(raw_df['GAME_DATE'])

if 'WL_NUM' not in raw_df.columns:
    raise ValueError("La columna WL_NUM no está disponible en el dataset.")

roll10_cols = sorted([col for col in raw_df.columns if col.startswith("ROLL10_")])
model_columns = ['GAME_DATE', 'WL_NUM'] + roll10_cols
model_df = raw_df[model_columns].copy()

print(f"Filas totales: {raw_df.shape[0]:,}")
print(f"Columnas originales: {len(raw_df.columns)}")
print(
    "Rango de fechas: "
    f"{model_df['GAME_DATE'].min().date()} → {model_df['GAME_DATE'].max().date()}"
)
print(f"Columnas usadas (ROLL10 + target): {len(model_columns)}")

display(
    pd.DataFrame({"Columnas seleccionadas": model_columns})
    .head(10)
)

initial_rows = model_df.shape[0]
model_df = model_df.dropna(subset=['WL_NUM'] + roll10_cols)
model_df['WL_NUM'] = model_df['WL_NUM'].astype(int)
raw_df = raw_df.loc[model_df.index].copy()

rows_dropped = initial_rows - model_df.shape[0]
print(f"Filas eliminadas por valores nulos: {rows_dropped}")

feature_cols = roll10_cols
trained_models = {}
metrics_df = pd.DataFrame()

## 4️⃣ Selector de fecha de corte

Elige una fecha de corte para entrenar con partidos previos (`Train`) y evaluar con partidos desde esa fecha en adelante (`Test`). Se muestran los tamaños y la proporción de victorias/derrotas por conjunto.

In [ ]:
def parse_date(value):
    if value is None:
        return None
    if isinstance(value, pd.Timestamp):
        return value
    if isinstance(value, dt.date):
        return pd.Timestamp(value)
    try:
        return pd.to_datetime(value)
    except Exception:
        return None

cutoff_default = model_df['GAME_DATE'].median() if not model_df.empty else pd.Timestamp('2025-02-15')
cutoff_widget = widgets.DatePicker(
    description='Fecha corte',
    value=cutoff_default.to_pydatetime().date() if pd.notnull(cutoff_default) else None,
)
cutoff_output = widgets.Output()

train_df = pd.DataFrame()
test_df = pd.DataFrame()
cutoff_date_value = None


def describe_split(df: pd.DataFrame) -> dict:
    total = len(df)
    wins = int(df['WL_NUM'].sum()) if total else 0
    losses = total - wins
    win_rate = wins / total if total else np.nan
    loss_rate = losses / total if total else np.nan
    return {
        'Partidos': total,
        'Victorias': wins,
        'Derrotas': losses,
        'Proporción W': win_rate,
        'Proporción L': loss_rate,
    }


def update_split(change=None):
    global train_df, test_df, cutoff_date_value
    cutoff = parse_date(cutoff_widget.value)
    with cutoff_output:
        cutoff_output.clear_output()
        if cutoff is None:
            print('Selecciona una fecha válida.')
            return
        cutoff_date_value = cutoff
        train_df = model_df[model_df['GAME_DATE'] < cutoff].copy()
        test_df = model_df[model_df['GAME_DATE'] >= cutoff].copy()

        summary = []
        for name, subset in [('Train', train_df), ('Test', test_df)]:
            info = describe_split(subset)
            info['Split'] = name
            summary.append(info)

        summary_df = pd.DataFrame(summary).set_index('Split')
        summary_df[['Proporción W', 'Proporción L']] = summary_df[
            ['Proporción W', 'Proporción L']
        ].applymap(lambda x: f"{x:.1%}" if pd.notnull(x) else '—')
        display(summary_df)
        if train_df.empty or test_df.empty:
            print('⚠️ Ajusta la fecha para asegurar ejemplos en train y test.')


cutoff_widget.observe(update_split, names='value')
display(cutoff_widget, cutoff_output)
update_split()

## 5️⃣ Entrenamiento del modelo

Se entrena una regresión logística (con escalado) y un Gradient Boosting Classifier usando las columnas `ROLL10_` como features y `WL_NUM` como objetivo.

In [ ]:
if train_df.empty or test_df.empty:
    raise ValueError('Ajusta la fecha de corte para disponer de datos en entrenamiento y prueba.')

X_train = train_df[feature_cols].to_numpy()
y_train = train_df['WL_NUM'].to_numpy()
X_test = test_df[feature_cols].to_numpy()
y_test = test_df['WL_NUM'].to_numpy()

model_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=1000, random_state=42)),
])
model_lr.fit(X_train, y_train)

model_gb = GradientBoostingClassifier(random_state=42)
model_gb.fit(X_train, y_train)

if len(y_test) > 0:
    y_pred_lr = model_lr.predict(X_test)
    y_proba_lr = model_lr.predict_proba(X_test)[:, 1]
    y_pred_gb = model_gb.predict(X_test)
    y_proba_gb = model_gb.predict_proba(X_test)[:, 1]
else:
    y_pred_lr = np.array([])
    y_proba_lr = np.array([])
    y_pred_gb = np.array([])
    y_proba_gb = np.array([])

trained_models = {
    'LogisticRegression': {
        'model': model_lr,
        'pred': y_pred_lr,
        'proba': y_proba_lr,
    },
    'GradientBoostingClassifier': {
        'model': model_gb,
        'pred': y_pred_gb,
        'proba': y_proba_gb,
    },
}


def safe_metric(func, *args, **kwargs):
    try:
        return func(*args, **kwargs)
    except ValueError:
        return np.nan

## 6️⃣ Resultados y gráficas

A continuación se presentan las métricas, curvas ROC y de calibración, y las matrices de confusión para ambos modelos.

### 6.1 Tabla de métricas

In [ ]:
if len(y_test) == 0:
    print('No hay muestras en el conjunto de prueba. Ajusta la fecha de corte para evaluar el modelo.')
else:
    metrics_rows = []
    for name, info in trained_models.items():
        y_pred = info['pred']
        y_proba = info['proba']
        metrics_rows.append(
            {
                'Modelo': name,
                'Accuracy': accuracy_score(y_test, y_pred),
                'Balanced Accuracy': balanced_accuracy_score(y_test, y_pred),
                'ROC AUC': safe_metric(roc_auc_score, y_test, y_proba),
                'PR AUC': safe_metric(average_precision_score, y_test, y_proba),
                'Brier': brier_score_loss(y_test, y_proba),
                'Log Loss': safe_metric(log_loss, y_test, y_proba),
            }
        )
    metrics_df = pd.DataFrame(metrics_rows).set_index('Modelo')
    display(metrics_df)

### 6.2 Curvas ROC y calibración

In [ ]:
if len(y_test) == 0:
    print('No hay datos de prueba para generar curvas.')
else:
    has_two_classes = np.unique(y_test).size > 1
    if has_two_classes:
        fig, ax = plt.subplots(figsize=(7, 5))
        for name, info in trained_models.items():
            RocCurveDisplay.from_predictions(
                y_test,
                info['proba'],
                name=name,
                ax=ax,
            )
        ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Azar')
        ax.set_title('Curva ROC')
        ax.legend()
        plt.show()
    else:
        print('Se requiere al menos una victoria y una derrota en test para la curva ROC.')

    fig, ax = plt.subplots(figsize=(7, 5))
    for name, info in trained_models.items():
        CalibrationDisplay.from_predictions(
            y_test,
            info['proba'],
            n_bins=10,
            ax=ax,
            name=name,
        )
    ax.set_title('Curva de calibración')
    plt.show()

### 6.3 Matriz de confusión

In [ ]:
if len(y_test) == 0:
    print('No hay datos de prueba para calcular la matriz de confusión.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for ax, (name, info) in zip(axes, trained_models.items()):
        ConfusionMatrixDisplay.from_predictions(
            y_test,
            info['pred'],
            display_labels=['Derrota', 'Victoria'],
            cmap='Blues',
            colorbar=False,
            ax=ax,
        )
        ax.set_title(f'Matriz de confusión - {name}')
    plt.tight_layout()
    plt.show()

## 7️⃣ Predicción por fecha elegida

Selecciona una fecha (posterior a la de corte) y el modelo con el que quieres estimar la probabilidad de victoria de cada equipo en esa jornada.

In [ ]:
prediction_default = (cutoff_date_value + pd.Timedelta(days=14)) if cutoff_date_value is not None else model_df['GAME_DATE'].max()
prediction_widget = widgets.DatePicker(
    description='Fecha predicción',
    value=prediction_default.to_pydatetime().date() if pd.notnull(prediction_default) else None,
)
model_selector = widgets.ToggleButtons(
    options=['LogisticRegression', 'GradientBoostingClassifier'],
    description='Modelo:',
)
prediction_output = widgets.Output()


def update_predictions(change=None):
    with prediction_output:
        prediction_output.clear_output()
        if not trained_models:
            print('Entrena los modelos antes de generar predicciones.')
            return
        pred_date = parse_date(prediction_widget.value)
        if pred_date is None:
            print('Selecciona una fecha válida.')
            return
        if cutoff_date_value is not None and pred_date < cutoff_date_value:
            print('La fecha de predicción debe ser igual o posterior a la fecha de corte.')
            return
        mask = model_df['GAME_DATE'] == pred_date
        if mask.sum() == 0:
            print('No hay registros para la fecha seleccionada.')
            return
        selected = model_df.loc[mask]
        feature_matrix = selected[feature_cols]
        model_info = trained_models.get(model_selector.value)
        if model_info is None:
            print('Modelo no encontrado. Vuelve a entrenar.')
            return
        proba = model_info['model'].predict_proba(feature_matrix)[:, 1]
        team_labels = (
            raw_df.loc[mask, 'TEAM_ABBREVIATION']
            if 'TEAM_ABBREVIATION' in raw_df.columns
            else raw_df.loc[mask].index.astype(str)
        )
        result_df = pd.DataFrame(
            {
                'TEAM': team_labels.values,
                'GAME_DATE': selected['GAME_DATE'].dt.date.values,
                'Prob_Win': proba,
            }
        ).sort_values('Prob_Win', ascending=False)
        display(result_df.reset_index(drop=True))


prediction_widget.observe(update_predictions, names='value')
model_selector.observe(update_predictions, names='value')

display(model_selector, prediction_widget, prediction_output)
update_predictions()

## 8️⃣ Importancia de variables

Se muestran las 15 variables ROLL10_ más influyentes según los coeficientes de la regresión logística y las importancias del Gradient Boosting.

In [ ]:
if not trained_models:
    print('Entrena los modelos para calcular la importancia de variables.')
else:
    lr_model = trained_models['LogisticRegression']['model']
    gb_model = trained_models['GradientBoostingClassifier']['model']

    lr_coefs = lr_model.named_steps['logreg'].coef_.ravel()
    lr_series = pd.Series(lr_coefs, index=feature_cols)
    lr_top = lr_series.reindex(lr_series.abs().sort_values(ascending=False).head(15).index)

    gb_importance = pd.Series(gb_model.feature_importances_, index=feature_cols)
    gb_top = gb_importance.sort_values(ascending=False).head(15)

    display(
        pd.DataFrame(
            {
                'Coef LR': lr_top,
                '|Coef LR|': lr_top.abs(),
                'Importancia GB': gb_top,
            }
        )
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].barh(lr_top.index[::-1], lr_top.values[::-1], color='#1f77b4')
    axes[0].set_title('Top 15 coeficientes (LR)')
    axes[0].set_xlabel('Coeficiente')

    axes[1].barh(gb_top.index[::-1], gb_top.values[::-1], color='#ff7f0e')
    axes[1].set_title('Top 15 importancias (GB)')
    axes[1].set_xlabel('Importancia')

    plt.tight_layout()
    plt.show()

## 9️⃣ Conclusión y plan de mejoras

El modelo ofrece una referencia rápida sobre la predicción de victorias basándose únicamente en estadísticas ROLL10_. Próximos pasos recomendados:

1. Calibrar probabilidades con `CalibratedClassifierCV` y validación temporal.
2. Optimizar el umbral de decisión en lugar de fijarlo en 0.5.
3. Implementar validación temporal robusta con `TimeSeriesSplit` (rolling CV).
4. Afinar hiperparámetros mediante búsqueda (`GridSearchCV`/`RandomizedSearchCV`) sobre `C`, `learning_rate`, `max_depth`, etc.
5. Incorporar variables de contexto (descanso, viajes, lesiones, fuerza de equipo...).
6. Reducir ruido en ROLL10 aplicando suavizados como EWMA con `min_periods=5`.
7. Evaluar métricas alineadas al negocio (ROC/PR, lift, ROI simulado, etc.).
8. Programar reentrenos periódicos para seguir la deriva temporal (ej. semanalmente).
9. Profundizar en interpretabilidad usando `permutation_importance` y gráficos SHAP.